# 01 · Tensors: shapes, strides, views and einsum

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/pytorch/01_tensors_views.ipynb) &nbsp; 📖 Concept deck: [Tensors, strides and views](https://nilesh-salpe.github.io/visual-ai/pytorch/01-tensors/) &nbsp; ⏱️ Suggested time: **30 min**

**You will build:** how a tensor maps to memory with strides, when `view` works and when you need `reshape`/`contiguous`, broadcasting rules, and attention scores with `einsum` and batched matmul.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)
print("PyTorch", torch.__version__)

## Task 1: Strides of a contiguous tensor

A contiguous row-major tensor of shape (d₀, d₁, …, dₙ₋₁) has stride[i] = product of all later dimensions. Write `contiguous_strides(shape)` and compare with `torch.empty(shape).stride()`.

In [ ]:
def contiguous_strides(shape):
    # TODO: last stride is 1; each earlier stride multiplies by the next dimension
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_contiguous_strides(f):
    for shape in [(3, 4), (2, 3, 5), (7,), (2, 1, 4, 8)]:
        assert tuple(f(shape)) == torch.empty(shape).stride(), f"shape {shape}"

check("contiguous_strides", _test_contiguous_strides, contiguous_strides)

## Task 2: Split and merge attention heads

Multi-head attention reshapes activations of shape (B, T, C) into (B, H, T, C/H) and back.
Write `split_heads(x, H)` (use `view` then `transpose(1, 2)`) and `merge_heads(y)` (the inverse). `merge_heads` needs `.contiguous()` before `view`, because the transpose made the tensor non-contiguous (or use `reshape`, which copies only when needed). Return both functions.

In [ ]:
def split_heads(x, H):
    # TODO
    raise NotImplementedError

def merge_heads(y):
    # TODO
    raise NotImplementedError

def split_merge():
    return split_heads, merge_heads

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_split_merge(f):
    s, m = f()
    x = torch.randn(2, 5, 12)
    y = s(x, 3)
    assert y.shape == (2, 3, 5, 4), f"expected (B, H, T, C/H), got {tuple(y.shape)}"
    assert not y.is_contiguous(), "after transpose the tensor is a non-contiguous view"
    assert torch.equal(y[1, 2, 4], x[1, 4, 8:12]), "head h of token t holds channels h·D … (h+1)·D"
    assert torch.equal(m(y), x), "merge must invert split"

check("split_merge", _test_split_merge, split_merge)

## Task 3: Attention scores three ways

For queries and keys of shape (B, H, T, D), the score tensor is S[b, h, i, j] = Σ_d q[b,h,i,d]·k[b,h,j,d] / √D.
Return `(s_matmul, s_einsum)`: one computed with `q @ k.transpose(-2, -1)`, one with `torch.einsum("bhid,bhjd->bhij", …)`. Both must equal an explicit loop.

In [ ]:
def scores_einsum(q, k):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_scores_einsum(f):
    q, k = torch.randn(2, 3, 4, 8), torch.randn(2, 3, 5, 8)
    a, b = f(q, k)
    assert a.shape == (2, 3, 4, 5) and b.shape == (2, 3, 4, 5)
    ref = torch.zeros(2, 3, 4, 5)
    for i in range(4):
        for j in range(5):
            ref[:, :, i, j] = (q[:, :, i] * k[:, :, j]).sum(-1) / 8 ** 0.5
    assert torch.allclose(a, ref, atol=1e-5) and torch.allclose(b, ref, atol=1e-5)

check("scores_einsum", _test_scores_einsum, scores_einsum)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Strides of a contiguous tensor
def contiguous_strides_solution(shape):
    out, acc = [], 1
    for d in reversed(shape):
        out.append(acc); acc *= d
    return tuple(reversed(out))

check("contiguous_strides (reference)", _test_contiguous_strides, contiguous_strides_solution)

In [ ]:
# @title Solution for Task 2: Split and merge attention heads
def split_heads_s(x, H):
    B, T, C = x.shape
    return x.view(B, T, H, C // H).transpose(1, 2)

def merge_heads_s(y):
    B, H, T, D = y.shape
    return y.transpose(1, 2).contiguous().view(B, T, H * D)

def split_merge_solution():
    return split_heads_s, merge_heads_s

check("split_merge (reference)", _test_split_merge, split_merge_solution)

In [ ]:
# @title Solution for Task 3: Attention scores three ways
def scores_einsum_solution(q, k):
    D = q.shape[-1]
    return q @ k.transpose(-2, -1) / D ** 0.5, torch.einsum("bhid,bhjd->bhij", q, k) / D ** 0.5

check("scores_einsum (reference)", _test_scores_einsum, scores_einsum_solution)

---
## Experiments
**Views share memory.** Changing a view changes the original, and a transpose only swaps strides. Broadcasting stretches size-1 dimensions without copying (its stride is 0).

In [ ]:
x = torch.arange(12.).reshape(3, 4); v = x.t()
print("x.stride()", x.stride(), " x.t().stride()", v.stride(), " same storage:", x.data_ptr() == v.data_ptr())
v[0, 0] = 100; print("after editing the view, x[0, 0] =", x[0, 0].item())
b = torch.tensor([1., 2., 3., 4.]).expand(3, 4); print("expanded stride:", b.stride(), "(0 = broadcast without copying)")
try:
    v.view(12)
except RuntimeError as e:
    print("view on a transposed tensor fails:", str(e)[:70], "…")
print("reshape works (copies):", v.reshape(12)[:4])

---
## Takeaways
* A tensor is storage + shape + strides; `view`, `transpose`, `permute`, slicing and `expand` only change the metadata.
* `view` needs compatible strides; after a transpose, call `.contiguous()` or use `reshape`.
* `einsum` and batched `@` express attention in one line; both dispatch to the same matmul kernels.